# Reward function testing — CPU only, no training

This notebook tests aligntune's **reward functions in isolation** and simulates
**exactly how each RL trainer backend calls them** (GRPO, Unsloth-GRPO, PPO) —
without loading any model, downloading any dataset, or running `trainer.train()`.
Everything here runs on CPU in a few seconds.

It exists to answer two questions without needing a GPU:

1. Do the built-in reward functions behave correctly on edge cases (empty text,
   `None`, mismatched batch lengths, exceptions)?
2. If I plug a reward function (built-in or custom) into GRPO / Unsloth-GRPO / PPO,
   will it actually receive the right data and score correctly — or will it silently
   score 0 because of how that particular backend calls reward functions?

Question 2 is the important one: several real bugs were found this way (a reward
function scoring 0 for every sample in GRPO because the trainer passed the whole
batch's reference column to every single completion instead of the one matching value —
see Section 4) and have since been fixed in `aligntune/core/rl/reward_handler.py`,
`aligntune/backends/*/rl/grpo/grpo.py` and `.../ppo/ppo.py`.
This notebook is the regression check for that fix, and a template for testing any
*new* custom reward function the same way before you ever touch a GPU.

The full automated version of everything below lives in
`tests/test_reward_function_trainer_simulation.py` (run via `pytest`). This notebook
is for interactive exploration — see the actual reward values, print the intermediate
kwargs a reward function receives, etc.

In [ ]:
import sys, os, types

# Make `import aligntune` work whether this notebook is run from the repo root
# or from inside notebooks/, without requiring `pip install -e .` first.
for candidate in (os.path.abspath("src"), os.path.abspath(os.path.join("..", "src"))):
    if os.path.isdir(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("This entire notebook runs with CUDA available =", torch.cuda.is_available(), "-- no GPU is required.")

import aligntune
print("aligntune loaded from:", aligntune.__file__)

## 1. Built-in reward functions — quick sanity + edge cases

Direct unit tests against `aligntune.rewards.core`, no trainer involved.

In [ ]:
from aligntune.rewards.core import (
    RewardConfig, RewardType, LengthReward, CoherenceReward,
    MathCorrectnessReward, CodeSyntaxReward, CompositeReward,
)

def make_reward(cls, **params):
    return cls(RewardConfig(reward_type=RewardType.LENGTH, weight=1.0, params=params))

length_reward = make_reward(LengthReward, min_length=2, max_length=10)
print("normal text            :", length_reward.compute("one two three"))
print("empty string            :", length_reward.compute(""))
print("None (used to crash!)   :", length_reward.compute(None))
print("non-string int          :", length_reward.compute(12345))

math_reward = make_reward(MathCorrectnessReward)
print("\nno math in text         :", math_reward.compute("no math here"))
print("None (used to crash!)   :", math_reward.compute(None))

code_reward = make_reward(CodeSyntaxReward)
print("\nno code block           :", code_reward.compute("just prose"))
print("valid python block      :", code_reward.compute("```python\nprint('hi')\n```"))

## 2. `CompositeReward` — combining multiple reward functions

Covers the 3 aggregation modes (`mean`, `worst_case`, `uncertainty_weighted`) and confirms a bug in one reward function doesn't take down the whole ensemble.

In [ ]:
good = make_reward(LengthReward, min_length=0, max_length=100)

class BrokenReward(LengthReward):
    def compute(self, text, reference=None, **kwargs):
        raise RuntimeError("simulated bug inside a reward function")

broken = BrokenReward(RewardConfig(reward_type=RewardType.LENGTH, weight=1.0, params={}))

composite = CompositeReward(reward_functions=[good, broken])
score = composite.compute("some text here")
print("composite score with one broken sub-reward (isolated, doesn't crash):", score)

try:
    CompositeReward(reward_functions=[])
except ValueError as e:
    print("\nCompositeReward([]) now fails fast instead of silently always scoring 0:")
    print(" ->", e)

## 3. Writing a custom reward function — what actually works everywhere

aligntune has (at least) three different call conventions across backends:

| Backend | `self.reward_functions` shape | Custom function key | Call signature |
|---|---|---|---|
| GRPO / DAPO / GSPO / DR_GRPO (TRL) | flat list of callables | `function` **or** `reward_function` | `fn(text_or_first_param, **kwargs)`, kwargs sliced per-sample |
| Unsloth GRPO | list of `{"function", "weight", "name"}` dicts | `function` **or** `reward_function` | signature-driven single call (was a 5-pattern try/except cascade) |
| PPO (`FunctionBasedRewardModel`) | list of `{"function", ...}` dicts | `function` **or** `reward_function` | `fn(text)` — **exactly one positional arg, nothing else, ever** |

A reward function that needs more than one required positional argument, or that
crashes on `None`/empty input, will silently score 0 under PPO even though it works
fine under GRPO. `assert_reward_fn_is_cross_trainer_safe` below checks for exactly
that before you wire a new reward into any trainer.

In [ ]:
def universal_style_reward(text, reference=None, **kwargs) -> float:
    '''Recommended style: one positional text-like arg + **kwargs.'''
    if not text or not isinstance(text, str):
        return 0.0
    return 1.0 if reference and reference.strip() in text else 0.0

def es_style_reward(prompt, response, reference, **kwargs) -> float:
    '''3 required positional args, no defaults -- works under GRPO
    (TRL forwards every column as a kwarg the adapter can bind by name) but
    is a landmine under PPO, which only ever supplies ONE positional arg.'''
    if not response or not reference:
        return 0.0
    return 1.0 if reference.strip() in response else 0.0


def assert_reward_fn_is_cross_trainer_safe(fn):
    '''Run this against any NEW custom reward before wiring it into ANY
    trainer. No GPU/model/dataset needed. Returns a list of problems (empty
    if the function looks safe everywhere).'''
    import inspect as _inspect
    sig = _inspect.signature(fn)
    required_positional = [
        p for p in sig.parameters.values()
        if p.kind in (_inspect.Parameter.POSITIONAL_ONLY, _inspect.Parameter.POSITIONAL_OR_KEYWORD)
        and p.default is _inspect.Parameter.empty
    ]
    has_var_keyword = any(p.kind == _inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values())

    problems = []
    if not has_var_keyword:
        problems.append("does not accept **kwargs -- will TypeError on any unexpected column/kwarg")
    if len(required_positional) > 1:
        problems.append(
            f"has {len(required_positional)} required positional params "
            f"{[p.name for p in required_positional]} -- PPO only ever supplies ONE "
            "positional arg (the completion text); this will silently score 0 under PPO"
        )
    try:
        result = fn("this is a sample completion for testing purposes")
        if result is not None and not isinstance(result, (int, float)):
            problems.append(f"fn(text) alone returned non-numeric type {type(result)}")
    except Exception as e:
        problems.append(f"fn(text) alone raised {type(e).__name__}: {e}")
    for edge in ["", "   "]:
        try:
            fn(edge)
        except Exception as e:
            problems.append(f"fn({edge!r}) raised {type(e).__name__}: {e} (should degrade, not raise)")
    return problems


print("universal_style_reward:", assert_reward_fn_is_cross_trainer_safe(universal_style_reward) or "OK, safe everywhere")
print("es_style_reward       :", assert_reward_fn_is_cross_trainer_safe(es_style_reward))

## 4. The headline bug: realistic GRPO batch → used to score 0

This is the exact "sometimes I'm seeing 0 rewards in GRPO" repro. It builds a
**realistic multi-generation batch** the way TRL's `GRPOTrainer` actually shapes it
(2 unique prompts × 4 generations each = 8 completions, with the dataset's `answer`
column repeated/aligned to all 8 completions — GSM8K-style, column named `answer` not
`reference`), wires up the **real** `TRLGRPOTrainer.setup_rewards()` with the **real**
registry reward `MathVerifiableReward` (not a mock), and calls the real
`_combined_reward_function` exactly as TRL would.

Before the fix in `aligntune/core/rl/reward_handler.py`, **every one of these 8
completions scored 0.0** — even the 4 that have the objectively correct answer —
because the whole batch's `answer` list was passed to every single completion's
reward call instead of the one value that actually corresponded to it, and because
there was no fallback from `answer` → `reference` (the built-in reward's parameter
name). Both are fixed now; this cell should show `[1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0]`.

In [ ]:
from aligntune.backends.trl.rl.grpo.grpo import TRLGRPOTrainer

def build_trainer(trainer_cls, rewards_config):
    trainer = object.__new__(trainer_cls)
    trainer.config = types.SimpleNamespace(rewards=rewards_config)
    trainer.reward_functions = []
    trainer.setup_rewards()
    return trainer

# Realistic GRPO batch: 2 unique prompts x 4 generations = 8 completions
completions = [
    "Let's compute. The answer is 18", "Hmm, the answer is 20",   # prompt 1: correct, wrong
    "The answer is 18",                "the answer is 20",        # prompt 1: correct, wrong
    "So the answer is 3",              "the answer is 5",         # prompt 2: correct, wrong
    "the answer is 3",                 "the answer is 5",         # prompt 2: correct, wrong
]
prompts = ["What is 12+6?"] * 4 + ["What is 10-7?"] * 4
answer  = ["18"] * 4 + ["3"] * 4  # GSM8K-style column name: 'answer', not 'reference'

# GRPO reward functions are handed straight to TRL's GRPOTrainer as
# `reward_funcs=...` now -- TRL calls and combines them itself, so there's no longer
# an aligntune-owned `_combined_reward_function` dispatcher on this class (that
# combining step moved into the prepared reward functions themselves as part of the
# Aug-3 reward-handling unification). Call the prepared function the same way TRL
# does: the whole batch at once, not a per-sample loop.
grpo_trainer = build_trainer(TRLGRPOTrainer, [{"type": "math_verifiable", "params": {}}])
grpo_scores = grpo_trainer.reward_functions[0](
    prompts=prompts, completions=completions, completion_ids=[[1, 2, 3]] * 8, answer=answer
)
print("GRPO rewards:", grpo_scores)
assert grpo_scores == [1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0]

print("\nGRPO scores this realistic batch correctly.")


### 4b. What it looked like *before* the fix (isolated repro, not the patched code path)

To make the bug concrete without reverting the actual fix, this cell re-implements
just the two buggy lines exactly as they used to be, so you can see the failure mode
directly: the same `reference` batch list gets handed to every completion untouched.

In [ ]:
def old_buggy_sync_compute(completions, reward_funcs, **kwargs):
    '''Exact reproduction of the pre-fix _sync_compute_rewards loop.'''
    from aligntune.core.rl.reward_handler import call_reward_safely
    batch_rewards = []
    for completion in completions:  # BUG: kwargs below is never sliced per-sample
        rewards = []
        for reward_func in reward_funcs:
            try:
                reward = call_reward_safely(reward_func, completion, **kwargs)
                if isinstance(reward, (int, float)):
                    rewards.append(float(reward))
            except Exception:
                continue
        batch_rewards.append(sum(rewards) / len(rewards) if rewards else 0.0)
    return batch_rewards

def naive_reference_reward(text, reference=None, **kw):
    if not reference:
        return 0.0
    return 1.0 if reference.strip() in text else 0.0

old_scores = old_buggy_sync_compute(
    ["the answer is 42"], [naive_reference_reward], reference=["42"]
)
print("Old (buggy) behavior, single completion, reference=['42']:", old_scores, "<- should be 1.0, silently wrong")

from aligntune.core.rl.reward_handler import CommonRewardHandler

class FakeTrainer(CommonRewardHandler):
    def __init__(self, fns):
        self.reward_functions = fns

fixed_scores = FakeTrainer([naive_reference_reward])._combined_reward_function(
    ["the answer is 42"], prompts=["p"], reference=["42"]
)
print("Fixed behavior, same input:                         ", fixed_scores)

## 5. Unsloth GRPO — same reward, different call convention

Unsloth GRPO stores rewards as `{"function", "weight", "name"}` dicts and used to pick
a calling pattern via a 5-step try/except-TypeError cascade (`_call_reward_function`),
which could silently reinterpret a genuine bug inside a reward function as "wrong
calling pattern" and retry with fewer arguments. It's now signature-driven (reuses the
same adapter GRPO uses), so a reward needing both `test_cases` and `reference`
together actually gets both in one call.

In [ ]:
from aligntune.backends.unsloth.rl.grpo.grpo import UnslothGRPOTrainer

def build_unsloth_trainer(rewards_config):
    trainer = object.__new__(UnslothGRPOTrainer)
    trainer.config = types.SimpleNamespace(rewards=rewards_config)
    trainer.reward_functions = []
    trainer.setup_rewards()
    return trainer

seen = {}
def needs_test_cases_and_reference(text, test_cases=None, reference=None, **kw):
    seen["test_cases"] = test_cases
    seen["reference"] = reference
    return 1.0

unsloth_trainer = build_unsloth_trainer(
    [{"type": "custom", "params": {"function": needs_test_cases_and_reference}}]
)

# Unsloth GRPO reward_functions are now prepared the same way as TRL GRPO
# (see Section 4) -- a flat list of TRL-batch-callables, called with the whole
# batch at once rather than through an aligntune-owned dispatcher. Custom
# functions (like this one) are wrapped through the same per-sample-slicing
# adapter as registry rewards, so both kwargs still arrive together correctly.
unsloth_trainer.reward_functions[0](
    prompts=["p"], completions=["completion text"],
    reference=["expected"],    # column literally named 'reference'
    test_cases=["assert True"],
)
print("Both test_cases and reference now arrive together:", seen)


## 6. PPO — `FunctionBasedRewardModel` calls with ONE positional arg only

No trainer/model needed here either: a fake tokenizer stands in for the real one
(`batch_decode` just returns known strings), and we call the actual `forward()` +
`score()` pipeline PPO uses. `FunctionBasedRewardModel` never threads any
`reference`/dataset-column data through to reward functions at all -- it only ever
calls `reward_func(text)`. So `universal_style_reward` scoring 0 below is *expected,
graceful* behavior (it correctly detects it has no reference to compare against and
degrades to 0 instead of crashing). `es_style_reward` scoring 0 is different: it
*crashes* with `TypeError` (missing 2 required arguments) every single call, which is
silently swallowed by `FunctionBasedRewardModel._compute_reward`'s bare
`except Exception: continue` -- a real bug masked as a normal 0 reward.

In [ ]:
from aligntune.core.rl.function_based_reward_model import FunctionBasedRewardModel

class FakeTokenizer:
    def batch_decode(self, input_ids, **kw):
        return ["the answer is 42", "FOO is the answer"]

model = FunctionBasedRewardModel(
    reward_functions=[universal_style_reward],
    tokenizer=FakeTokenizer(), device="cpu", dtype=torch.float32,
)
input_ids = torch.zeros((2, 5), dtype=torch.long)
backbone_out = model.model.forward(input_ids)
rewards = model.score(backbone_out.hidden_states)
print("PPO reward tensor shape:", tuple(rewards.shape))
print("universal_style_reward under PPO:", rewards[:, 0, 0].tolist(),
      "<- expected 0: no reference is ever available under PPO, and the function degrades gracefully")

print("\nNow es_style_reward (3 required positional args) under the same PPO path:")
model2 = FunctionBasedRewardModel(
    reward_functions=[es_style_reward],
    tokenizer=FakeTokenizer(), device="cpu", dtype=torch.float32,
)
backbone_out2 = model2.model.forward(input_ids)
rewards2 = model2.score(backbone_out2.hidden_states)
print("es_style_reward under PPO:", rewards2[:, 0, 0].tolist(),
      "<- these are CRASHES (TypeError) silently swallowed, not legitimate zero scores")

## 7. The `function` vs `reward_function` key — now portable everywhere

GRPO/Unsloth-GRPO historically only recognized `params["function"]` for a custom
reward; PPO only recognized `params["reward_function"]`. Reusing one spec across
backends used to mean PPO silently substituted a generic length-based fallback reward
instead of raising an error. Both keys are now accepted everywhere.

In [ ]:
from aligntune.backends.trl.rl.ppo.ppo import TRLPPOTrainer

def build_ppo_trainer(rewards_config):
    trainer = object.__new__(TRLPPOTrainer)
    trainer.config = types.SimpleNamespace(rewards=rewards_config)
    trainer.reward_functions = []
    trainer.setup_rewards()
    return trainer

# The exact same spec shape used for GRPO/Unsloth-GRPO above ({"function": fn})
ppo_trainer = build_ppo_trainer([{"type": "custom", "params": {"function": universal_style_reward}}])
print("Loaded function is the custom one, not a fallback:",
      ppo_trainer.reward_functions[0]["function"] is universal_style_reward)
print("Reward name:", ppo_trainer.reward_functions[0]["name"])

## 8. Run the full automated regression suite

Everything demonstrated above (plus many more edge cases) is captured as pytest
tests in `tests/test_reward_function_trainer_simulation.py`. Run it any time you
touch reward-function code or add a new trainer backend.

In [ ]:
import subprocess, sys as _sys

repo_root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
result = subprocess.run(
    [_sys.executable, "-m", "pytest", "tests/test_reward_function_trainer_simulation.py", "-q"],
    cwd=repo_root, capture_output=True, text=True,
)
print(result.stdout[-3000:])
print(result.stderr[-1000:])

## Summary

| # | Bug | Fixed in |
|---|---|---|
| 1 | Batch-shaped `reference`/`answer`/etc. kwargs weren't sliced per completion — every reward call got the whole batch list instead of its own value, silently scoring 0 | `aligntune/core/rl/reward_handler.py` (`slice_batch_kwargs_for_sample`) |
| 2 | No fallback from `answer`/`solution`/`ground_truth`/etc. → `reference` in the TRL-mixin path (Unsloth already had one) | `aligntune/core/rl/reward_handler.py` (`resolve_reward_call_kwargs`) |
| 3 | Custom-reward key differed by backend (`function` vs `reward_function`) — same spec silently dropped under PPO | `grpo.py` (TRL + Unsloth), `ppo.py` |
| 4 | Unsloth's calling-pattern cascade could mask real bugs inside a reward function and lost `test_cases`+`reference` together when the column was named `reference` | `unsloth/rl/grpo/grpo.py` (`_call_reward_function`, now signature-driven) |
| 5 | Silent exception swallowing at debug-level only, everywhere | `reward_handler.py` (now `logger.warning` with function name + sample index) |
| 6 | `LengthReward`/`MathCorrectnessReward`/`CodeSyntaxReward`/`CoherenceReward` crashed on `None`/non-string input | `rewards/core.py` |
| 7 | `batch_compute` silently truncated the batch via `zip()` on length mismatch | `rewards/core.py` |
| 8 | `CompositeReward([])` constructed silently; `uncertainty_weighted` mode didn't normalize std-dev correctly | `rewards/core.py` |
| 9 | `RubricReward.batch_compute` crashed on a numpy-array `completions` input (`if not completions` before the numpy→list conversion) | `rewards/rubric_reward.py` |

No GPU, model download, or dataset download was used anywhere in this notebook.